In [ ]:
# papermill parameters
COLLECTION  = "OEB"     # "OEB" or "OE"
WORK_ROOT   = "/work"
SAVE        = False     # preview first; set True to write the feature tables
PREVIEW_ROWS = 12
RUN_CONTEXT = "manual"


In [ ]:
# === features.ipynb — retrieval-oriented feature prep ===
#
# Consumes the normalised tables saved by data.ipynb and derives the fields the retrieval
# baselines index. It does not alter the canonical normalisation; it only builds extra views.
#
# What changed in S1 (D-022): paths come from utils.run_context.data_paths(COLLECTION), the
# feature functions live in utils.feature_prep where they are tested against the OEB tables
# they originally produced, and the same pipeline is applied to the synthetic query sets —
# which the notebook previously had no way to reach, so no lexical method could be run on a
# BC3CAT-Syn query at all.
#
# Inputs   {collection}_short_norm.parquet, {collection}_long_norm.parquet,
#          {collection}_{queryset}_norm.parquet
# Outputs  {collection}_short_feats.parquet, {collection}_long_feats.parquet,
#          {collection}_{queryset}_feats.parquet, {collection}_features_meta.json

import json
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path(WORK_ROOT) / "src"):
    if (candidate / "utils").is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import pandas as pd

from utils.feature_prep import build_features
from utils.run_context import data_paths

pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

paths = data_paths(COLLECTION, work_root=WORK_ROOT)

REQUIRED = [
    "item_key", "text_word", "text_char", "numbers", "tokens_word",
    "parameters_norm", "param_values_multi_norm",
]

MUST_HAVE = [
    "param_phrases", "text_word_phrases_add", "text_word_phrases_replace",
    "tokens_bigram", "text_word_uni_bi", "has_numbers", "tokens_word_bi",
]


def featurise(path: Path, name: str) -> pd.DataFrame:
    frame = pd.read_parquet(path)
    missing = [c for c in REQUIRED if c not in frame.columns]
    if missing:
        raise KeyError(f"{name}: required columns missing from {path.name}: {missing}")

    feat = build_features(frame)

    absent = [c for c in MUST_HAVE if c not in feat.columns]
    if absent:
        raise KeyError(f"{name}: feature pipeline did not produce: {absent}")

    print(f"{name:16} rows={len(feat):6} | columns={len(feat.columns)}")
    return feat


short_feat = featurise(paths.short_norm, "short (resumen)")
long_feat = featurise(paths.long_norm, "long (texto)")
query_feats = {
    name: featurise(path, name) for name, path in sorted(paths.query_norm.items()) if path.exists()
}

if paths.query_norm and not query_feats:
    raise FileNotFoundError(
        f"{COLLECTION} has query sets {sorted(paths.query_norm)} but none is normalised yet. "
        "Run data.ipynb with SAVE=True first."
    )

# --- preview ---
def preview(df: pd.DataFrame, name: str, n: int = 12) -> None:
    cols = [c for c in ("item_key", "text_word", "text_word_params", "param_phrases", "has_numbers") if c in df.columns]
    print(f"\n--- {name} ---")
    print(df[cols].head(n).to_string(index=False))

preview(short_feat, "short (resumen)", PREVIEW_ROWS)
preview(long_feat, "long (texto)", PREVIEW_ROWS)
for name, frame in query_feats.items():
    preview(frame, name, PREVIEW_ROWS)

print("\nPreview generated. To write the tables, set SAVE=True and re-run.")

# --- SAVE (single switch) ---
if SAVE:
    written = []
    short_feat.to_parquet(paths.short_feats, index=False)
    long_feat.to_parquet(paths.long_feats, index=False)
    written += [paths.short_feats, paths.long_feats]

    for name, frame in query_feats.items():
        frame.to_parquet(paths.query_feats[name], index=False)
        written.append(paths.query_feats[name])

    features_meta = {
        "collection": COLLECTION,
        "text_fields": {
            "raw_text": "text",
            "word_unigram": "text_word",
            "word_unigram_phrases_add": "text_word_phrases_add",
            "word_unigram_phrases_replace": "text_word_phrases_replace",
            "char_3_5": "text_char",
            "word_unigram_params": "text_word_params",
        },
        "token_fields": {
            "bm25_unigram": "tokens_word",
            "bm25_unigram_bigram": "tokens_word_bi",
        },
        "aux_fields": {
            "numbers": "numbers",
            "tokens_word": "tokens_word",
            "has_numbers": "has_numbers" if "has_numbers" in short_feat.columns else None,
            "param_phrases": "param_phrases",
            "param_tokens": "param_tokens",
        },
    }
    with open(paths.features_meta, "w", encoding="utf-8") as f:
        json.dump(features_meta, f, ensure_ascii=False, indent=2)
    written.append(paths.features_meta)

    print("\nSaved:")
    for path in written:
        print(f"   {path} ({path.stat().st_size:,} bytes)")
else:
    print("Not saving (SAVE is False). Set SAVE=True to write files.")
